# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassan9039/intern1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row represents one content item for one client on one report date. This is the grain of the fact_content_daily_performance table.

Time window: I will develop my analysis using March 2026 as the main development month. Features must represent information available before the prediction/decision moment, while future performance used for the label or proxy must remain separate. I will not use the June 2026 _sample table for developing label logic because it represents the final month and should be treated as a sealed test period.

In [2]:
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{token}'
)
""")

REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

result = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_grain_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {REL}
""").df()

print(result)

duplicates = con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
FROM {REL}
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("\nDuplicate grain rows:")
print(duplicates)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_grain_rows first_date  last_date
0     9841378            9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Duplicate grain rows:
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, row_count]
Index: []


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_engaged_sessions, and ga4_sessions because these are page-level performance signals available in the warehouse before the decision moment.
Label / proxy: A future content-performance outcome that represents whether a page is a content-refresh opportunity. It will not be used as a feature.

Context: client_hash_id, content_hash_id, and report_date because they identify or group observations and help with time-based analysis, but should not be used by the model as predictive features.

Excluded: Future performance information and label-derived fields are excluded from the features because they would not be known at the decision moment and could cause data leakage.

In [7]:
features = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM {REL}
WHERE gsc_data_available IS TRUE
LIMIT 10
""").df()

features

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280,<NA>,<NA>
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,7.832461,<NA>,<NA>
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,3.272727,<NA>,<NA>
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,5.636364,<NA>,<NA>
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,4.500000,<NA>,<NA>


gsc_impressions — available when the page's Google Search Console data has been collected for the reporting date.

gsc_clicks — available when the page's Google Search Console data has been collected for the reporting date.

gsc_avg_position — available when the page's Google Search Console data has been collected for the reporting date.

ga4_sessions — available when GA4 data is available for the client and reporting date; otherwise it is missing and must not be treated as zero.

ga4_engaged_sessions — available when GA4 data is available for the client and reporting date; otherwise it is missing and must not be treated as zero.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Grain: Checks whether any report_date + client_hash_id + content_hash_id combination appears more than once.

In [14]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {REL}
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


Count and date span: Checks the number of March 2026 rows and confirms the reporting period.

In [16]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {REL}
""").df()

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


Availability: Checks how many rows have usable GSC and GA4 data using the corresponding availability flags.

In [18]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM {REL}
""").df()

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


**Leakage experiment**


I deliberately include a future outcome as a feature to demonstrate data leakage. I compare the resulting score with the honest feature set, then remove the leaked feature and keep the honest result.

In [17]:
march = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {REL}
WHERE gsc_data_available IS TRUE
""").df()

april_rel = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""

april = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    CASE WHEN SUM(gsc_clicks) > 0 THEN 1 ELSE 0 END AS label
FROM {april_rel}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

df = march.merge(april, on=["client_hash_id", "content_hash_id"], how="inner")

df["leaked_label"] = df["label"]

df[["gsc_impressions", "gsc_clicks", "gsc_avg_position", "label", "leaked_label"]].head()

,gsc_impressions,gsc_clicks,gsc_avg_position,label,leaked_label
0,20,0,3.350000,1,1
1,1,0,0.000000,0,0
2,125,1,4.928000,1,1
3,7,0,4.000000,0,0
4,11,0,2.272727,1,1


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_honest = df[["gsc_impressions", "gsc_clicks", "gsc_avg_position"]].fillna(0)
X_leaked = df[["gsc_impressions", "gsc_clicks", "gsc_avg_position", "leaked_label"]].fillna(0)
y = df["label"]

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X_honest, y, test_size=0.2, random_state=42
)

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaked, y, test_size=0.2, random_state=42
)

honest_model = DecisionTreeClassifier(max_depth=4, random_state=42)
leaked_model = DecisionTreeClassifier(max_depth=4, random_state=42)

honest_model.fit(Xh_train, yh_train)
leaked_model.fit(Xl_train, yl_train)

honest_score = accuracy_score(yh_test, honest_model.predict(Xh_test))
leaked_score = accuracy_score(yl_test, leaked_model.predict(Xl_test))

print("Honest score:", round(honest_score, 3))
print("Leaked score:", round(leaked_score, 3))

Honest score: 0.777
Leaked score: 1.0


Leakage result: The honest model achieved 0.777 accuracy, while adding the future label-derived leaked_label produced a perfect 1.000 accuracy. This unrealistic jump demonstrates data leakage. The leaked column is removed from the final feature set, and the honest 0.777 result is retained.

In [20]:
final_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

print("Final features:", final_features)
print("Leaked feature included:", "leaked_label" in final_features)

Final features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
Leaked feature included: False


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation: GA4 availability varies across the warehouse. In March 2026, only 413,966 of 9,841,378 rows had ga4_data_available IS TRUE. Therefore, GA4-based features cannot be assumed to represent every content item, which may limit the usefulness of GA4 signals for some clients.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.